# 22 · Un'API REST che autentica con il certificato di ADR

Il certificato di ADR apre l'Hub (notebook 04) e un broker MQTT (notebook 21). Ora apriamo **un'API nostra**: nessuna chiave da custodire, nessun token da rinnovare.
L'API è un file solo, `api\cert-api.cs`, e gira **su questo computer** (`https://localhost:8443`): non serve infrastruttura in più.

**Cosa controlla** l'API, a ogni connessione:
1. la catena arriva alla **Root CA di ADR**, la sola di cui si fida;
2. il certificato **non è revocato**: ADR pubblica la lista di revoca (CRL) e l'indirizzo è scritto nel certificato;
3. il certificato è nelle sue date ed è per l'autenticazione di un client.
Il device è identificato dal **nome comune** del certificato (il registration ID): il corpo del messaggio non deve dire chi manda.

**Serve:** i notebook 01 e 02a-02b eseguiti. Il device che chiama l'API, `api-client-01`, è fatto apposta per questo notebook: lo registra lui, e riceve da ADR un certificato vero. Per la parte sulla revoca serve anche il notebook 09. Serve .NET 11 (`dotnet`).

**Da dire.** "Il device ha già un certificato. Il mio servizio non deve inventare un'altra credenziale: si fida della CA che già c'è."

In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

In [ ]:
$cliente = $customDevices | Where-Object Id -eq 'api-client-01'     # il client dell'API: un device fatto apposta per questo notebook
$cartella = Initialize-DemoDevice -Device $cliente -OutDir $out -DpsName $dps.Name -ResourceGroup $rg -Namespace $adrName
$api = Join-Path $base '..\api\cert-api.cs'
$dev = [pscustomobject]@{ Dir = $cartella }      # la cartella del device: da qui si leggono certificato e chiave
"API: $api | client: $($cliente.Id)"


## 1 · Le CA di ADR, e l'API in marcia
Alla API diamo **il file con le due CA di ADR** (la Root e la Issuing): sono pubbliche. La Root è l'àncora di fiducia, la Issuing serve a completare la catena.
La prima volta `dotnet` compila il file: ci vogliono alcuni secondi.

In [ ]:
$ca = Export-AdrCaCertificates -ChainPath (Join-Path $cartella 'issued-chain.pem') -OutDir (Join-Path $out 'api')
$proc = Start-DemoCertApi -ApiFile $api -TrustFile $ca.Trust
"API in ascolto su https://localhost:8443 (processo $($proc.Id)). Si fida della Root di ADR e della Issuing $($ca.IcaSubject)."

## 2 · Chi sono, secondo il mio certificato
`-SkipCertificateCheck`: il certificato **dell'API** è uno usa e getta per `localhost`, che il computer non conosce. Serve a proteggere il canale: è il **client** che deve dimostrare chi è.

In [ ]:
$adr = New-DeviceClientCertificate $dev
Invoke-RestMethod 'https://localhost:8443/whoami' -Certificate $adr -SkipCertificateCheck | Format-List | Out-String

## 3 · Un messaggio, senza dire chi lo manda
Il messaggio di telemetria non contiene il nome del device in un campo di sicurezza: l'API lo ricava dal certificato.

In [ ]:
$msg = New-DeviceTelemetry ([pscustomobject]@{ Id = 'api-client-01' }) | ConvertTo-Json -Compress
Invoke-RestMethod 'https://localhost:8443/telemetry' -Method Post -Certificate $adr -SkipCertificateCheck -ContentType 'application/json' -Body $msg | ConvertTo-Json -Compress

## 4 · Chi non entra
- il **certificato di bootstrap** dello stesso device: firmato da un'altra CA;
- **nessun certificato**: l'API lo chiede nella stretta di mano TLS, e senza non si connette.

In [ ]:
$boot = New-DeviceClientCertificate $dev -CertFile 'bootstrap-cert.pem' -KeyFile 'bootstrap-key.pem'
try { Invoke-RestMethod 'https://localhost:8443/whoami' -Certificate $boot -SkipCertificateCheck; "ACCETTATO (non doveva)" } catch { "bootstrap: rifiutato -> $($_.Exception.Message)" }
try { Invoke-RestMethod 'https://localhost:8443/whoami' -SkipCertificateCheck; "ACCETTATO (non doveva)" } catch { "senza certificato: rifiutato -> $($_.Exception.Message)" }

## 5 · Un certificato revocato in ADR
Il device `device-da-revocare` ha avuto il suo certificato **revocato** nel notebook 09 (`az iot adr ns device auth revoke-certs`). Le date sono ancora valide, ma l'API legge la lista di revoca di ADR e lo rifiuta.
Il motivo lo scrive l'API nel suo log (`Revoked`).

In [ ]:
$rev = Join-Path $out '09-device-revoca\device-da-revocare'
if (-not (Test-Path (Join-Path $rev 'issued-chain.pem'))) { "Manca il certificato revocato: esegui prima il notebook 09." }
else {
    $revocato = New-DeviceClientCertificate ([pscustomobject]@{ Dir = $rev })
    try { Invoke-RestMethod 'https://localhost:8443/whoami' -Certificate $revocato -SkipCertificateCheck; "ACCETTATO (non doveva: la revoca e' gia' nella lista?)" }
    catch { "certificato revocato: rifiutato -> $($_.Exception.Message)" }
    "--- Cosa ha detto l'API:"
    Get-Content (Join-Path ([IO.Path]::GetTempPath()) 'cert-api.log') | Where-Object { $_ -match '^(accepted|refused)' }
}

## Pulizia
Ferma l'API. `dotnet run` avvia un processo figlio: `/T` ferma entrambi.

In [ ]:
& taskkill /PID $proc.Id /T /F 2>&1 | Out-Null
"API fermata."

## Cosa abbiamo visto
- Un'API **nostra** può autenticare con il certificato che ADR emette al device: nessuna chiave condivisa, nessun token.
- Il certificato dice **chi è il device** (nome comune = registration ID). L'API non ha un elenco di device: si fida della **CA**.
- **La revoca vale anche qui**, ma solo se l'API la controlla. ADR pubblica la **lista di revoca (CRL)**: l'indirizzo è nel certificato, e il certificato revocato dal notebook 09 ci compare con le sue date di revoca. Un servizio che non guarda la CRL accetterebbe un certificato revocato.

**Due cose trovate provando** (valgono per qualunque servizio che non sia di Microsoft):
- Il certificato del device ha un'**estensione critica privata di Microsoft** (`1.3.6.1.4.1.311.133.1.1.1`). Un validatore X.509 rigoroso deve rifiutare un'estensione critica che non conosce, e .NET lo fa. L'API la tollera **per nome** e rifiuta qualunque altra.
- Con la sola Issuing CA come àncora .NET non costruisce la catena: serve la **Root** come àncora, con la Issuing come aiuto.

**Non provato:** un servizio non .NET (nginx, OpenSSL) con questi certificati; la revoca fatta *durante* una connessione già aperta.